<a href="https://colab.research.google.com/github/AmeliaHumphrey/Wildfires2.0/blob/main/Wildfires.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import datasets and data cleaning

In [3]:
import os
from sklearn import tree
from sklearn import metrics
from sklearn.model_selection import train_test_split
import pandas as pd
weather_path = os.path.join("/content/4419504.csv")
weather = pd.read_csv(weather_path)
fire_path = os.path.join("/content/InFORM_FireOccurrence_Public_-904755089098175112.csv")
fires = pd.read_csv(fire_path)


In [4]:
#Cleaning the wildfire data
#filter for only wildfires aka not human started
wf = fires[fires["Incident Type Category"] == "WF"].copy()
wf["DATE"] = pd.to_datetime(
    wf["Fire Discovery Date Time"],
    format = "%m/%d/%Y %I:%M:%S %p",
    errors="coerce").dt.date # missing dates become NAT

#convert data into desired format
wf["DATE"] = pd.to_datetime(wf["DATE"])

#chosing columns to keep
wf = wf[[
    "POO County", "POO State", "Incident Name", "Incident Type Category",
    "Incident Size", "Initial Latitude", "Initial Longitude", "DATE",
    "FireOutDateTime"
]]


#filter for the april to october as that is went wildfire conditions are relevant

wf = wf[wf["DATE"].dt.month.isin([6,7,8,9,10])]



#for multiple fires on a day but them in the same row
wf = wf.groupby("DATE").agg({
    #combine unique values into one comma separateed string
    "POO County": lambda x: ", ".join(x.unique().astype(str)),
    "POO State": lambda x: ", ".join(x.unique().astype(str)),
    "Incident Name": lambda x: ", ".join(x.unique().astype(str)),
    "Incident Type Category": lambda x: ", ".join(x.unique().astype(str)),
    "Incident Size" : lambda x: ", ".join(x.unique().astype(str)),
    "Initial Latitude" : lambda x: ", ".join(x.unique().astype(str)),
    "Initial Longitude" : lambda x: ", ".join(x.unique().astype(str)),
    "FireOutDateTime" : lambda x: ", ".join(x.unique().astype(str))
}).reset_index() #fix the indices



#add the column that hold the amount of fires that occured that day
count_wf = fires[fires["Incident Type Category"] == "WF"].copy()
count_wf["DATE"] = pd.to_datetime(
    count_wf["Fire Discovery Date Time"],
    format = "%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
    ).dt.date # missing dates become NAT
count_wf["DATE"] = pd.to_datetime(count_wf["DATE"])
count_wf = count_wf[ count_wf["DATE"].dt.month.isin([6,7,8,9,10])]
count_wf = count_wf.groupby("DATE").size().reset_index(name="fire_count")

#merge fire count with original wildfire data
wf = wf.merge(count_wf, on="DATE", how="left")


#cleaning the weather data
climate = weather.drop(columns=["TAVG"]).copy()
climate["DATE"] = pd.to_datetime(climate["DATE"])


#combine the two datasets
combine_data = climate.merge(wf, on="DATE", how="left")
#missing fires get zeros
combine_data["fire_count"] = combine_data["fire_count"].fillna(0)

#column for if there are any fires for easy checking
combine_data["Fire"] = (combine_data["fire_count"]> 0)

#just may to october
combine_data = combine_data[combine_data["DATE"].dt.month.isin([6,7,8,9,10])]

combine_data.head()



,STATION,DATE,AWND,PRCP,TMAX,TMIN,POO County,POO State,Incident Name,Incident Type Category,Incident Size,Initial Latitude,Initial Longitude,FireOutDateTime,fire_count,Fire
0,USW00024131,2017-06-01,5.82,0.04,70,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,False
1,USW00024131,2017-06-02,5.82,0.00,81,51,Ada,US-ID,"OTR 63, OTR 64, OTR 65, OTR 66",WF,"0.1, 0.3, 0.5","43.265, 43.2606, 43.2633, 43.2689","-116.0778, -116.0767, -116.0822, -116.1803",6/5/2017 5:59:59 AM,4.0,True
2,USW00024131,2017-06-03,9.40,0.00,94,62,Ada,US-ID,OTR 67,WF,0.5,43.2672,-116.0761,6/6/2017 5:59:59 AM,1.0,True
3,USW00024131,2017-06-04,10.74,0.00,88,56,"Ada, Elmore",US-ID,"OTR 68, OTR 69, OTR 70, OTR 71, OTR 72, OTR 73...",WF,"1.0, 0.5, 3.0, 0.8, 4.0, 0.1, 5.0, 2.0, 6.0","43.2564, 43.2603, 43.2533, 43.0767, 43.2578, 4...","-116.1567, -116.1597, -116.1994, -116.1572, -1...","6/6/2017 5:59:59 AM, 6/7/2017 5:59:59 AM",11.0,True
4,USW00024131,2017-06-05,6.93,0.00,77,50,"Elmore, Ada",US-ID,"MM119 I84, OTR 79",WF,"715.0, 0.5","42.96698, 43.2617","-115.34099, -116.1642","6/10/2017 5:59:59 AM, 6/7/2017 5:59:59 AM",2.0,True


Making the features
- making a 7 day preciptation
- making a 3 day max temperature
- making a 7 day max temperature
- making a column for how long it was since the last rain


In [6]:
data = combine_data.sort_values(by="DATE").reset_index(drop=True).copy()
data["year"] = data["DATE"].dt.year
data["month"] = data["DATE"].dt.month
data["dayOfYear"] = data["DATE"].dt.dayofyear
#need to not cross year because average will be incorrect thus use rolling
data["tmax_3d"] = data.groupby("year")["TMAX"].rolling(3, min_periods=1).mean().reset_index(0, drop=True)
data["tmax_7d"] = data.groupby("year")["TMAX"].rolling(7, min_periods=1).mean().reset_index(0, drop=True)
data["prcp_7d"] = data.groupby("year")["PRCP"].rolling(7, min_periods=1).mean().reset_index(0, drop=True)

#last day since there was precipitation
data["since_rain"] = (data["PRCP"] > 0.01).groupby(data["year"]).cumsum()
data["last_rain"] = data.groupby(["year","since_rain"]).cumcount()

#the feature columns
features = ["tmax_3d", "tmax_7d", "prcp_7d", "last_rain", "AWND", "PRCP", "TMIN", "TMAX"]




Splitting the Data
70/15/15

In [7]:
from sklearn.model_selection import train_test_split
x = data[features]
y = data["Fire"].astype(int)
#adding weight to see if weight the days will more fires gives a more accurate prediction
weights = 1  + data["fire_count"]

print(x.shape)
print(y.shape)

#stratify so that a split does not end up with only wildfires
filler_x, test_x, filler_y, test_y, filler_w, test_w = train_test_split(x, y, weights, test_size=0.15, random_state=0, stratify=y)
#print the shapes of the train data to test the display
train_x, val_x, train_y, val_y, train_w, val_w = train_test_split(filler_x, filler_y, filler_w, test_size=0.15/0.85, random_state=0, stratify=filler_y)
print(train_x.shape, val_x.shape, test_x.shape)
print(train_y.shape, val_y.shape, test_y.shape)
#

(1519, 8)
(1519,)
(1063, 8) (228, 8) (228, 8)
(1063,) (228,) (228,)


Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression

#scaling data for better results
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_xs = scaler.fit_transform(train_x)
val_xs = scaler.transform(val_x)
modelLR = LogisticRegression(max_iter = 1000)
modelLR.fit(train_xs, train_y)
accuracy_lr = modelLR.score(val_xs, val_y)


modelLRw = LogisticRegression(max_iter = 1000)
modelLRw.fit(train_xs, train_y, sample_weight=train_w)
accuracy_lrw = modelLRw.score(val_xs, val_y)


print("Baseline:", accuracy_lr, " weighted ", accuracy_lrw)
print("Recall baseline:", recall_score(val_y, modelLR.predict(val_xs)),
      "weighted:", recall_score(val_y, modelLRw.predict(val_xs)))
print("Precision baseline:", precision_score(val_y, modelLR.predict(val_xs)),
      "weighted:", precision_score(val_y, modelLRw.predict(val_xs)))
print("ROC AUC baseline:", roc_auc_score(val_y, modelLR.predict_proba(val_xs)[:, 1]),
      "weighted:", roc_auc_score(val_y, modelLRw.predict_proba(val_xs)[:, 1]))



Baseline: 0.6271929824561403  weighted  0.5175438596491229
Recall baseline: 0.37362637362637363 weighted: 0.9010989010989011
Precision baseline: 0.5483870967741935 weighted: 0.44808743169398907
ROC AUC baseline: 0.6558915537017727 weighted: 0.664073153124248


Decision Tree

In [9]:
from sklearn import tree
from sklearn import metrics
from sklearn.metrics import recall_score, precision_score, roc_auc_score

#Baseline
#default settings
model = tree.DecisionTreeClassifier()
model.fit(train_x, train_y)
accuracy = model.score(val_x, val_y)

#more specific
model2 = tree.DecisionTreeClassifier(max_depth=6)
model2.fit(train_x, train_y)
accuracy2 = model2.score(val_x, val_y)

#weighted

modelw = tree.DecisionTreeClassifier()
modelw.fit(train_x, train_y, sample_weight=train_w)
accuracyw = modelw.score(val_x, val_y)

#more specific
model2w = tree.DecisionTreeClassifier(max_depth=6)
model2w.fit(train_x, train_y, sample_weight=train_w)
accuracy2w = model2w.score(val_x, val_y)

print("Baseline:", accuracy, " weighted ", accuracyw)
print("Baseline:", accuracy2, " weighted ", accuracy2w)

print("Recall baseline:", recall_score(val_y, model2.predict(val_x)),
      "weighted:", recall_score(val_y, model2w.predict(val_x)))
print("Precision baseline:", precision_score(val_y, model2.predict(val_x)),
      "weighted:", precision_score(val_y, model2w.predict(val_x)))
print("ROC AUC baseline:", roc_auc_score(val_y, model2.predict_proba(val_x)[:, 1]),
      "weighted:", roc_auc_score(val_y, model2w.predict_proba(val_x)[:, 1]))

Baseline: 0.5877192982456141  weighted  0.6403508771929824
Baseline: 0.6140350877192983  weighted  0.5087719298245614
Recall baseline: 0.4065934065934066 weighted: 0.8131868131868132
Precision baseline: 0.5211267605633803 weighted: 0.4378698224852071
ROC AUC baseline: 0.6128980508542552 weighted: 0.6169487446859709


Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, roc_auc_score

modelRF = RandomForestClassifier(n_estimators=300, min_samples_leaf=5)
modelRF.fit(train_x, train_y)
accuracyRF = modelRF.score(val_x, val_y)
print("Baseline:", accuracyRF)

modelRFw = RandomForestClassifier(n_estimators=300, min_samples_leaf=5)
modelRFw.fit(train_x, train_y, sample_weight = train_w)
accuracyRFw = modelRFw.score(val_x, val_y)
print("Weighted:", accuracyRFw)

print("Recall baseline:", recall_score(val_y, modelRF.predict(val_x)),
      "weighted:", recall_score(val_y, modelRFw.predict(val_x)))
print("Precision baseline:", precision_score(val_y, modelRF.predict(val_x)),
      "weighted:", precision_score(val_y, modelRFw.predict(val_x)))
print("ROC AUC baseline:", roc_auc_score(val_y, modelRF.predict_proba(val_x)[:, 1]),
      "weighted:", roc_auc_score(val_y, modelRFw.predict_proba(val_x)[:, 1]))

print(pd.Series(modelRF.feature_importances_, index=features).sort_values(ascending=False))



Baseline: 0.631578947368421
Weighted: 0.5877192982456141
Recall baseline: 0.42857142857142855 weighted: 0.7692307692307693
Precision baseline: 0.5492957746478874 weighted: 0.48951048951048953
ROC AUC baseline: 0.6610251062805808 weighted: 0.6665597176546082
tmax_7d      0.184524
tmax_3d      0.177828
TMAX         0.153660
last_rain    0.140687
TMIN         0.118580
AWND         0.108929
prcp_7d      0.101306
PRCP         0.014486
dtype: float64


Testing

In [34]:

from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
test_pred = modelRF.predict(test_x)
test_proba = modelRF.predict_proba(test_x)[:, 1]
print("Test Accuracy", accuracy_score(test_y, test_pred))
print("Test Recall", recall_score(test_y, test_pred))
print("Test Precision", precision_score(test_y, test_pred, zero_division=0))
print("Test ROC AUC", roc_auc_score(test_y, test_proba))
print("Always-no-fire baseline", 1 - test_y.mean())

Test Accuracy 0.631578947368421
Test Recall 0.38461538461538464
Test Precision 0.5555555555555556
Test ROC AUC 0.6864522338974894
Always-no-fire baseline 0.6008771929824561
